# cfDNA Fragmentomics — Exploratory Data Analysis

**Goal:** Build confidence that a real biological signal exists in the data *before* training any model.  
If EDA looks wrong — fix the data, not the model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import sys, pathlib

ROOT = pathlib.Path().resolve().parent
sys.path.insert(0, str(ROOT))

df = pd.read_csv(ROOT / 'data/processed/features_model.csv')
print('Shape:', df.shape)
print(df['label'].value_counts())

---
## Step 1 — Class Distribution

**Why this matters:**  
You have ~1.78× more healthy samples than cancer. This is a class imbalance.  

A naive model that *always predicts healthy* would score **64% accuracy** — and be completely useless for detecting cancer. Accuracy is the wrong metric here. This is why we use **AUC-ROC** and why models need `class_weight='balanced'` or equivalent.

Check this first. Always.

In [ ]:
counts = df['label'].value_counts()
ratio  = counts['healthy'] / counts['cancer']

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(counts.index, counts.values,
              color=['#2196F3', '#F44336'], width=0.5, edgecolor='white')

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of samples')
ax.set_ylim(0, counts.max() * 1.15)
ax.spines[['top','right']].set_visible(False)
ax.text(0.98, 0.95, f'Imbalance ratio: {ratio:.2f}:1',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=10, color='gray')
plt.tight_layout()
plt.savefig(ROOT / 'results/eda_01_class_distribution.png', dpi=150)
plt.show()

print(f'Healthy : {counts["healthy"]}  ({100*counts["healthy"]/len(df):.1f}%)')
print(f'Cancer  : {counts["cancer"]}  ({100*counts["cancer"]/len(df):.1f}%)')
print(f'Ratio   : {ratio:.2f}:1')
print()
print('Implication: use AUC-ROC as primary metric, not accuracy.')
print('All models will need class_weight="balanced" or equivalent.')

---
## EDA Summary

| Check | Result | Verdict |
|---|---|---|
| Class balance | 119 healthy / 67 cancer (1.78:1) | Use AUC, not accuracy. Apply `class_weight='balanced'` |
| Histogram overlay | Cancer has +3.4pp more mass in 100–150bp window | Biology confirmed ✓ |
| SFR direction | Cancer 0.281 vs Healthy 0.226 | Correct direction ✓ |
| Mann-Whitney p-value | p = 1.46e-08 | Difference is not random ✓ |
| PCA | 89% variance in 2D. Cancer drifts left along PC1 (fragment shortness axis) | Partial separation, overlap in centre |
| UMAP | Healthy forms tight arc. Cancer is scattered — no single cluster | Signal is real but heterogeneous |

**Key findings:**
1. **The signal is real.** Fragment length distributions differ significantly between cancer and healthy (p < 1e-7). This is the DELFI biology — nucleosome disruption in cancer releases more short sub-nucleosomal fragments.
2. **Healthy is consistent, cancer is heterogeneous.** Healthy std = 0.048, cancer std = 0.078. Different cancer patients produce different degrees of disruption depending on type, stage, and tumour fraction.
3. **Partial overlap will limit model performance.** Some cancer samples are indistinguishable from healthy in this feature space. This is expected for chr1-only features.
4. **Expected AUC: 0.78–0.88.** The gap from Cristiano et al. (~0.94) is explained by chr1-only scope and smaller sample size (186 vs 498).

**Modelling decisions locked in by EDA:**
- Primary metric → **AUC-ROC**
- All models → `class_weight='balanced'`
- Feature sets → bins + scalars (18) for XGBoost/RF/MLP; hist (311) for CNN
- Evaluation → 5-fold stratified cross-validation + held-out test set
- Permutation test → **500 shuffles** to confirm signal is not noise (pre-committed, not added after seeing results)

### UMAP Interpretation

**Healthy forms a coherent arc (bottom-left, UMAP1: 1–4, UMAP2: 1–5)**  
Blue dots cluster together in a tight crescent. This reflects what we saw in the statistics — healthy people have consistent, predictable cfDNA profiles. Nucleosome positioning is stable across individuals, so their fragment length distributions look similar to each other.

**Cancer is scattered across the upper and right regions**  
Red dots have no single tight cluster — they are spread across the entire plot. This is not a data quality problem. It is a biological reality: cancer is heterogeneous. Different patients have different cancer types, different stages, and different tumour-fraction levels in their blood. Each produces a different degree of nucleosome disruption, and therefore a different fragment profile. There is no single "cancer signature" — only a deviation from healthy in varying directions and magnitudes.

**Separation compared to PCA**  
UMAP does not show dramatically cleaner separation than PCA. The bottom-left is mostly healthy, but the rest of the space is heavily mixed. This confirms the overlap is genuine — it is not a PCA limitation. The feature space has real ambiguity between some cancer and healthy samples.

**Modelling implications**  
- High **specificity** is achievable — the healthy cluster is compact and learnable  
- **Sensitivity** will be harder — many cancer samples sit in healthy-looking regions of the space  
- Expect AUC in the **0.78–0.88** range, not 0.95+  
- The gap from Cristiano et al. (AUC ~0.94) is expected: they used whole-genome 5Mb windowed features on 498 samples; we use chr1-only with 186 samples  

**Bottom line:** The signal is real, statistically significant (p = 1.46e-08), and visible in the data. The partial overlap means models need to be evaluated on AUC and sensitivity/specificity — not accuracy.

In [ ]:
import umap

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
embedding = reducer.fit_transform(X_scaled)   # X_scaled already defined in Step 4

colors = {'cancer': '#F44336', 'healthy': '#2196F3'}
fig, ax = plt.subplots(figsize=(7, 6))

for label, color in colors.items():
    mask = y == label
    ax.scatter(embedding[mask, 0], embedding[mask, 1],
               c=color, label=f'{label.capitalize()} (n={mask.sum()})',
               alpha=0.75, edgecolors='white', linewidths=0.4, s=55)

ax.set_xlabel('UMAP 1', fontsize=12)
ax.set_ylabel('UMAP 2', fontsize=12)
ax.set_title('UMAP — bins + scalars (18 features)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(ROOT / 'results/eda_05_umap.png', dpi=150)
plt.show()

---
## Step 5 — UMAP

**Why this matters:**  
PCA found partial separation — cancer drifts left, healthy drifts right, with overlap in the middle. But PCA is **linear**: it can only draw straight projection axes. If the true boundary between cancer and healthy is curved or non-linear, PCA undersells the separation.

UMAP (Uniform Manifold Approximation and Projection) is allowed to bend and fold the feature space. It tries to preserve the **neighbourhood structure** of your data — samples that are similar in 18-dimensional space should be close together in 2D. Samples that are different should be far apart.

**What to look for:**
- If UMAP shows cleaner clusters than PCA → non-linear structure exists → XGBoost/MLP will outperform logistic regression
- If UMAP looks the same as PCA → the data is genuinely linearly structured
- If UMAP shows cancer splitting into sub-groups → possible cancer subtypes with different fragment profiles

**Key parameter — `n_neighbors`:** Controls how local vs global the structure is. Small values (5–10) emphasise tight local clusters. Large values (30–50) emphasise global layout. We use 15 as a balanced default.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 18 features: 13 bins + 5 scalars
bin_cols    = [c for c in df.columns if c.startswith('bin_')]
scalar_cols = ['short_long_ratio', 'mean_length', 'median_length', 'mono_peak_height', 'peak_to_flank']
feat_cols   = bin_cols + scalar_cols

X = df[feat_cols].astype(float).values
y = df['label'].values

# scale before PCA — PCA is sensitive to feature magnitude
X_scaled = StandardScaler().fit_transform(X)

pca  = PCA(n_components=2, random_state=42)
pcs  = pca.fit_transform(X_scaled)
var  = pca.explained_variance_ratio_

# plot
colors = {'cancer': '#F44336', 'healthy': '#2196F3'}
fig, ax = plt.subplots(figsize=(7, 6))

for label, color in colors.items():
    mask = y == label
    ax.scatter(pcs[mask, 0], pcs[mask, 1],
               c=color, label=f'{label.capitalize()} (n={mask.sum()})',
               alpha=0.7, edgecolors='white', linewidths=0.4, s=55)

ax.set_xlabel(f'PC1 ({var[0]*100:.1f}% variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({var[1]*100:.1f}% variance)', fontsize=12)
ax.set_title('PCA — bins + scalars (18 features)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(ROOT / 'results/eda_04_pca.png', dpi=150)
plt.show()

print(f'PC1 explains {var[0]*100:.1f}% of variance')
print(f'PC2 explains {var[1]*100:.1f}% of variance')
print(f'Together   : {sum(var)*100:.1f}%')
print()
print('Top features driving PC1 (largest absolute loading):')
loadings = pd.Series(pca.components_[0], index=feat_cols).abs().sort_values(ascending=False)
print(loadings.head(5).round(4).to_string())

---
## Step 4 — PCA (Principal Component Analysis)

**Why this matters:**  
Steps 2 and 3 looked at features one at a time. PCA asks: *when you consider all features together*, do cancer and healthy samples occupy different regions of the feature space?

PCA works by finding the directions of maximum variance in your data and projecting every sample onto those directions. PC1 captures the most variance, PC2 the second most. You plot every sample as a dot (PC1, PC2), coloured by label.

**What to look for:**
- **Clear separation** → the combined feature space distinguishes the two groups. A linear model will do well.
- **Partial separation** → non-linear models (XGBoost, MLP) needed.
- **Complete overlap** → no structure in the data. Go back and fix features.

**Important:** PCA is linear. It will underestimate separation if the true boundary between classes is curved. That's why we follow with UMAP in Step 5.

We use the **18 features** (13 bins + 5 scalars) here — not the 311-histogram columns. Bins and scalars are what the tree models and MLP will train on, so PCA on those is the most relevant diagnostic.

In [ ]:
from scipy.stats import mannwhitneyu

cancer_sfr  = df[df['label'] == 'cancer']['short_long_ratio'].astype(float)
healthy_sfr = df[df['label'] == 'healthy']['short_long_ratio'].astype(float)

stat, p = mannwhitneyu(cancer_sfr, healthy_sfr, alternative='greater')

fig, ax = plt.subplots(figsize=(5, 5))

bp = ax.boxplot(
    [healthy_sfr.values, cancer_sfr.values],
    labels=['Healthy', 'Cancer'],
    patch_artist=True,
    medianprops=dict(color='white', linewidth=2),
    flierprops=dict(marker='o', markersize=4, alpha=0.5),
    widths=0.45,
)
bp['boxes'][0].set_facecolor('#2196F3')
bp['boxes'][1].set_facecolor('#F44336')

ax.set_ylabel('Short-to-Long Ratio (SFR)', fontsize=12)
ax.set_title('SFR Distribution — Cancer vs Healthy', fontsize=13, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)

p_text = f'p = {p:.2e}' if p < 0.001 else f'p = {p:.4f}'
sig     = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
ax.text(0.98, 0.97, f'Mann-Whitney\n{p_text}  {sig}',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig(ROOT / 'results/eda_03_sfr_boxplot.png', dpi=150)
plt.show()

print(f'Cancer  SFR — median: {cancer_sfr.median():.4f}  mean: {cancer_sfr.mean():.4f}  std: {cancer_sfr.std():.4f}')
print(f'Healthy SFR — median: {healthy_sfr.median():.4f}  mean: {healthy_sfr.mean():.4f}  std: {healthy_sfr.std():.4f}')
print(f'\nMann-Whitney U = {stat:.0f},  p = {p:.4e}')
print(f'Significant (p<0.05): {p < 0.05}')

---
## Step 3 — Short-to-Long Ratio Boxplot + Mann-Whitney Test

**Why this matters:**  
The histogram overlay showed group *averages*. Averages hide variance — a few extreme outliers can make two overlapping distributions look separated.

The boxplot shows the **full distribution** of SFR values for every individual sample:
- The box = middle 50% of samples (25th–75th percentile)
- The line inside = median
- The whiskers = rest of the range
- Dots = outliers

**Mann-Whitney U test** then asks a precise question: if you randomly pick one cancer sample and one healthy sample, what is the probability that the cancer sample has a higher SFR? If p < 0.05, the separation is not random chance.

This is the step where you move from *"it looks different"* to *"it is statistically different."*

In [ ]:
hist_cols = [c for c in df.columns if c.startswith('hist_')]
bp_values = [int(c.split('_')[1]) for c in hist_cols]

cancer_mean  = df[df['label'] == 'cancer'][hist_cols].mean()
healthy_mean = df[df['label'] == 'healthy'][hist_cols].mean()

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(bp_values, healthy_mean.values, color='#2196F3', lw=1.8, label='Healthy (n=119)')
ax.plot(bp_values, cancer_mean.values,  color='#F44336', lw=1.8, label='Cancer  (n=67)')

# shade short window and mark nucleosome peak
ax.axvspan(100, 150, alpha=0.08, color='#F44336', label='Short window (100–150bp)')
ax.axvline(167, color='gray', lw=1, ls='--', alpha=0.7)
ax.text(168, ax.get_ylim()[1]*0.92, '167bp\n(mono-nuc peak)', fontsize=8, color='gray', va='top')

ax.set_xlabel('Fragment length (bp)', fontsize=12)
ax.set_ylabel('Mean normalised frequency', fontsize=12)
ax.set_title('Mean Fragment Length Distribution — Cancer vs Healthy', fontsize=13, fontweight='bold')
ax.set_xlim(90, 400)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(ROOT / 'results/eda_02_histogram_overlay.png', dpi=150)
plt.show()

# numeric gate: cancer must have more mass in the short window
short_cancer  = cancer_mean[[c for c in hist_cols if 100 <= int(c.split('_')[1]) <= 150]].sum()
short_healthy = healthy_mean[[c for c in hist_cols if 100 <= int(c.split('_')[1]) <= 150]].sum()
print(f'Short-window mass (100–150bp):')
print(f'  Cancer  : {short_cancer:.4f}')
print(f'  Healthy : {short_healthy:.4f}')
print(f'  Cancer excess: {short_cancer - short_healthy:+.4f}  <- must be positive')

---
## Step 2 — Mean Fragment Length Histogram Overlay

**Why this matters:**  
This is the biological signal made visible.

Healthy cfDNA is wrapped around nucleosomes (~167bp per nucleosome). This **protects** that length of DNA from degradation, creating a sharp peak at 167bp in the fragment length distribution.

In cancer, nucleosome positioning is disrupted at active oncogene regions. This releases more **sub-nucleosomal short fragments (100–150bp)**. The 167bp peak flattens slightly and there is a visible shoulder of extra mass in the short-fragment region.

**What to look for:**
- Both curves peak near 167bp
- Cancer curve sits **higher** than healthy in the 100–150bp region
- Healthy curve sits **higher** near the 167bp peak

If the two curves are indistinguishable — the signal is not there. If they separate — the model has something real to learn.